# RAKSHAK — Cross-Dataset Validation (CICIDS2017 → UNSW-NB15)

Everything trained and evaluated so far has used splits of the *same* dataset (CICIDS2017) — that measures how well the model fits CICIDS2017's specific traffic, capture tool, and label distribution, not whether it learned something that generalizes to network intrusion detection more broadly.

This notebook runs the real test: train a model once on CICIDS2017, using only the schema **common** to both datasets (`data/processed/cicids_common.parquet` / `unsw_common.parquet`, built by `preprocess.py`'s `build_cicids_common_features()`/`build_unsw_common_features()` — 6 features that measure the same underlying quantities in both datasets, since CICIDS2017's 25 selected features don't exist in UNSW-NB15 at all). Then evaluate that *same, never-retrained* model in two places:

1. **In-distribution**: CICIDS2017's own held-out test split (common-schema version) — the baseline.
2. **Cross-dataset (zero-shot)**: the entirety of UNSW-NB15's common-schema data — data the model has never seen, captured with a different tool (Argus, not CICFlowMeter), with a different class balance (see `04_unsw_eda.ipynb`).

The gap between those two is the actual generalization measurement.

Deliberate simplifications versus the main pipeline (this is a validation study, not a production model):
- **Single Random Forest**, not the full 3-model ensemble.
- **No scaling** — Random Forest splits on raw values, so MinMaxScaler wouldn't change its behavior (unlike XGBoost/LightGBM's gradient-based fitting, tree splits are scale-invariant).
- **`class_weight="balanced"` instead of SMOTE** — after finding that SMOTE-oversampled U2R got badly overfit by LightGBM during Colab tuning (see `PROGRESS.md`), this smaller side-experiment avoids that same risk by handling imbalance through loss weighting instead of synthetic rows.

## 1. Load both common-schema datasets

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RANDOM_STATE = 42
TEST_SIZE = 0.2

cicids = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "cicids_common.parquet")
unsw = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "unsw_common.parquet")

print(f"CICIDS2017 common: {cicids.shape}")
print(f"UNSW-NB15 common:  {unsw.shape}")
cicids.head()

CICIDS2017 common: (2519994, 8)
UNSW-NB15 common:  (153684, 8)


,duration,fwd_packets,bwd_packets,fwd_bytes,bwd_bytes,fwd_bwd_ratio,bytes_per_sec,Label
0,0.000003,2,0,12,0,2.0,12000.0,Normal
1,0.000109,1,1,6,6,0.5,12000.0,Normal
2,0.000052,1,1,6,6,0.5,12000.0,Normal
3,0.000034,1,1,6,6,0.5,12000.0,Normal
4,0.000003,2,0,12,0,2.0,12000.0,Normal


## 2. Train once on CICIDS2017 (common schema), split for an in-distribution baseline

In [2]:
X_cicids = cicids.drop(columns=["Label"])
y_cicids = cicids["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_cicids, y_cicids, test_size=TEST_SIZE, stratify=y_cicids, random_state=RANDOM_STATE
)

rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)

print("Trained on:", X_train.shape)

Trained on: (2015995, 7)


## 3. Evaluate: in-distribution (CICIDS2017 held-out test)

In [3]:
y_pred_cicids = rf.predict(X_test)

print("=== In-distribution: CICIDS2017 held-out test ===")
print(classification_report(y_test, y_pred_cicids, digits=4))

in_dist_weighted_f1 = f1_score(y_test, y_pred_cicids, average="weighted")
in_dist_macro_f1 = f1_score(y_test, y_pred_cicids, average="macro")
print(f"Weighted F1: {in_dist_weighted_f1:.4f}")
print(f"Macro F1:    {in_dist_macro_f1:.4f}")

=== In-distribution: CICIDS2017 held-out test ===
              precision    recall  f1-score   support

         DoS     0.9838    0.9949    0.9893     64334
      Normal     0.9984    0.9587    0.9781    418870
       Probe     0.9915    0.9992    0.9953     18139
         R2L     0.2943    0.8632    0.4389      2259
         U2R     0.0307    0.9144    0.0595       397

    accuracy                         0.9643    503999
   macro avg     0.6597    0.9461    0.6922    503999
weighted avg     0.9923    0.9643    0.9770    503999

Weighted F1: 0.9770
Macro F1:    0.6922


**Observations:** *(rewrite in your own words)*

- Weighted F1 0.9770, macro F1 0.6922 — solid on the big classes (DoS/Normal/Probe all >0.97 F1) even with only 6 features, but U2R's precision collapses to 0.0307 (recall stays high at 0.9144). With this few features to work with, the model over-predicts U2R heavily — it can find *something* different about U2R flows, but not precisely enough to avoid a lot of false alarms.
- This confirms the 6-feature schema is a real handicap next to the full 25-feature ensemble (which reached U2R F1 ~0.43-0.65 depending on tuning) — expected, since this is intentionally a smaller, common-denominator feature set for comparability, not the production model.

## 4. Evaluate: cross-dataset, zero-shot (all of UNSW-NB15)

Same model object, no retraining, no fitting of any kind on UNSW-NB15 — just `.predict()`.

In [4]:
X_unsw = unsw.drop(columns=["Label"])
y_unsw = unsw["Label"]

y_pred_unsw = rf.predict(X_unsw)

print("=== Cross-dataset (zero-shot): full UNSW-NB15 ===")
print(classification_report(y_unsw, y_pred_unsw, digits=4))

cross_weighted_f1 = f1_score(y_unsw, y_pred_unsw, average="weighted")
cross_macro_f1 = f1_score(y_unsw, y_pred_unsw, average="macro")
print(f"Weighted F1: {cross_weighted_f1:.4f}")
print(f"Macro F1:    {cross_macro_f1:.4f}")

=== Cross-dataset (zero-shot): full UNSW-NB15 ===


/opt/anaconda3/envs/ids_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/ids_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/ids_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize

              precision    recall  f1-score   support

         DoS     0.0000    0.0000    0.0000     11630
      Normal     0.5573    0.9999    0.7157     85646
       Probe     0.0000    0.0000    0.0000     28546
         R2L     0.0000    0.0000    0.0000     26250
         U2R     0.0000    0.0000    0.0000      1612

    accuracy                         0.5572    153684
   macro avg     0.1115    0.2000    0.1431    153684
weighted avg     0.3106    0.5572    0.3988    153684

Weighted F1: 0.3988
Macro F1:    0.1431


**Observations:** *(rewrite in your own words)*

- Total collapse: weighted F1 drops from 0.9770 to **0.3988**, macro F1 from 0.6922 to **0.1431**. The model predicts "Normal" for essentially every single row (99.99% recall on Normal, exactly 0.0 precision/recall on all 4 other classes) — it isn't "somewhat worse" at detecting UNSW-NB15's attacks, it has effectively stopped trying.
- This is a genuinely severe generalization failure, not a mild one — worth investigating *why* before concluding "the model doesn't generalize," since a few different explanations could produce this same symptom. Section 5 checks the most likely one.

## 5. Why did it collapse this hard? Comparing feature value distributions

A Random Forest's splits are hard numeric thresholds learned from the *training* data's value ranges (e.g. "if `fwd_bytes` > 66, go left"). If the two datasets' "same-named" features actually sit on very different numeric scales — plausible here, since CICIDS2017 (CICFlowMeter) and UNSW-NB15 (Argus) use different flow-capture tools with different timeout/aggregation conventions — nearly every row from the *other* dataset could fall on the same side of *every* learned split, collapsing almost all predictions into whichever single leaf that represents.

In [5]:
comparison = pd.DataFrame(
    {
        "CICIDS2017 median": X_cicids.median(),
        "UNSW-NB15 median": X_unsw.median(),
    }
)
comparison["ratio (UNSW / CICIDS)"] = (
    comparison["UNSW-NB15 median"] / comparison["CICIDS2017 median"]
).round(2)
comparison

,CICIDS2017 median,UNSW-NB15 median,ratio (UNSW / CICIDS)
duration,0.050645,0.387087,7.64
fwd_packets,2.000000,10.000000,5.00
bwd_packets,2.000000,8.000000,4.00
fwd_bytes,66.000000,980.000000,14.85
bwd_bytes,156.000000,554.000000,3.55
fwd_bwd_ratio,0.666667,1.111111,1.67
bytes_per_sec,3711.839367,24549.326574,6.61


**Observations:** *(rewrite in your own words)*

- UNSW-NB15's typical flow is systematically larger across nearly every shared feature: median duration ~7.6x longer, median forward packets 5x more, median forward bytes ~15x more, median throughput (`bytes_per_sec`) ~6.6x higher.
- This directly supports the threshold-mismatch explanation from the cell above: the two datasets' "same-named, same-quantity" features occupy genuinely different numeric neighborhoods, most likely due to differing flow-capture/aggregation conventions between CICFlowMeter and Argus (and possibly different underlying traffic-generation setups when each dataset was originally created) — not a bug in `preprocess.py`'s common-schema mapping itself.
- Practical implication for the report: a shared feature *name* and a shared feature *definition* ("forward bytes" in both cases) does not guarantee a shared feature *distribution*. Genuine cross-dataset transfer would need something scale-invariant — e.g. per-dataset normalization, or a model family less sensitive to absolute thresholds — not just matching column names. This is a legitimate, citable limitation, not a failure to hide.

## 6. Summary

| | Weighted F1 | Macro F1 |
|---|---|---|
| In-distribution (CICIDS2017 test) | 0.9770 | 0.6922 |
| Cross-dataset (UNSW-NB15, zero-shot) | 0.3988 | 0.1431 |

**Conclusion** *(rewrite in your own words)*: RAKSHAK's models are highly accurate *within* the dataset they were trained on, but this experiment shows that accuracy does not transfer to a differently-captured dataset without adaptation — even when using features chosen specifically to be common between them. The root cause traced to a genuine, measurable distribution shift in the underlying feature values (Section 5), not a modeling mistake. Worth stating plainly in the final report as an honest limitation and a direction for future work (e.g. per-dataset feature normalization, or domain-adaptation techniques), rather than omitting it or overstating RAKSHAK's general-purpose readiness.